# Laya server (Kaggle + Cloudflare tunnel)

Run all cells. The last cell prints `LAYA_URL` and `LAYA_TOKEN` and then keeps running (that is what keeps the link alive).
Settings: Accelerator = GPU (CPU also works, slower), Internet = On.

The URL is public but every request needs the random token. A new URL and token are made each time this notebook starts.
Do not share this notebook's output publicly.

In [ ]:
%pip install -q laya fastapi uvicorn

In [ ]:
%%writefile server.py
"""Laya inference server. Runs on Kaggle (NOT on the developer's computer), behind a Cloudflare tunnel.

POST /analyze  (Authorization: Bearer $LAYA_TOKEN)  {"headlines": [...]}
GET  /health   (no auth)
"""
import hmac
import os
import threading

os.environ.setdefault("USE_TF", "0")  # model card: avoids a TensorFlow deadlock when loading Laya

import laya
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

TOKEN = os.environ["LAYA_TOKEN"]
MODEL = os.environ.get("LAYA_MODEL", "convaiinnovations/laya")
agent = laya.load(MODEL)
lock = threading.Lock()  # one prediction at a time

QUESTIONS = {
    "sentiment": {
        "type": "choice",
        "instructions": "What is the likely effect of this financial news on the stock price?",
        "criteria": {
            "bearish": "negative for the stock price",
            "bullish": "positive for the stock price",
            "neutral": "no clear effect, or purely informational",
        },
    },
    "kind": {
        "type": "choice",
        "instructions": "Is this headline a concrete company event, or generic filler?",
        "criteria": {
            "event": "reports a concrete event: earnings, guidance, a deal, a rating change, a product or legal or regulatory action involving the company",
            "filler": "generic explainer, stock-quote page, promotion, or fund-holding boilerplate",
        },
    },
}


class Req(BaseModel):
    headlines: list[str] = Field(max_length=200)


app = FastAPI(title="Laya server")


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL}


@app.post("/analyze")
def analyze(req: Req, authorization: str | None = Header(default=None)):
    if not authorization or not hmac.compare_digest(authorization, f"Bearer {TOKEN}"):
        raise HTTPException(401, "bad token")
    results = []
    with lock:
        for h in req.headlines:
            a = agent.predict(h[:500], QUESTIONS)["answers"]
            results.append({
                "sentiment": a["sentiment"]["choice"],
                "sentiment_probs": a["sentiment"]["probabilities"],
                "kind": a["kind"]["choice"],
                "kind_probs": a["kind"]["probabilities"],
            })
    return {"model": MODEL, "results": results}

In [ ]:
import os, re, secrets, subprocess, sys, time, urllib.request

PORT = 8765
TOKEN = secrets.token_urlsafe(24)   # new random password every run; the tunnel URL is public, this is what protects it
env = dict(os.environ, LAYA_TOKEN=TOKEN, USE_TF="0")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", str(PORT)],
                          env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)

def up(url):
    try:
        urllib.request.urlopen(url, timeout=5)
        return True
    except Exception:
        return False

t0 = time.time()
while not up(f"http://127.0.0.1:{PORT}/health"):
    if server.poll() is not None:
        print(open("server.log").read()[-3000:])
        raise SystemExit("model server crashed (log above)")
    if time.time() - t0 > 600:
        raise SystemExit("model server did not start within 10 minutes")
    time.sleep(3)
print(f"model server up after {time.time() - t0:.0f}s")

cf = "/tmp/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, 0o755)
tun = subprocess.Popen([cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                       stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
if not URL:
    print(open("tunnel.log").read()[-2000:])
    raise SystemExit("no tunnel URL")
for _ in range(30):   # wait until the public URL actually answers
    if up(URL + "/health"):
        break
    time.sleep(2)

print("=" * 60)
print("LAYA_URL=" + URL)
print("LAYA_TOKEN=" + TOKEN)
print("=" * 60)
print("Put both lines in Backend/.env. Keep this cell running; stopping the notebook closes the link.")

end = time.time() + 11 * 3600
beat = 0
while time.time() < end:
    if server.poll() is not None or tun.poll() is not None:
        print("a process died; restart the notebook")
        break
    time.sleep(60)
    beat += 1
    if beat % 30 == 0:
        print(f"still serving, {beat // 60}h{beat % 60:02d}m")